# DS-01 · Property Price Estimator
**Team:** _add names_   |   **Date:** 30 September 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.max_columns", 50)

# Figures are saved to reports/figures (repo layout) so they can be used in the README and slides
FIG_DIR = Path("../reports/figures") if Path("../data").exists() else Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
def save_fig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

def find_data(name):
    """Look in data/ first; in Colab without the repo, show a Choose Files button."""
    for p in [Path("../data") / name, Path("data") / name, Path(name)]:
        if p.exists():
            return p
    try:
        from google.colab import files
        print(f"Please upload {name}")
        up = files.upload()
        return Path(list(up.keys())[0])
    except ImportError:
        raise FileNotFoundError(f"Put {name} in the data/ folder")

path = find_data("house_price_data.csv")
df_raw = pd.read_csv(path)
print("Loaded", path, "→ shape", df_raw.shape)

## 1. Problem statement
**Client:** a real-estate agency in Dhaka, Chattogram, Sylhet and Rajshahi. Prices are set by hand and two agents often quote the same house lakhs apart. They want a data-driven first estimate.

**Business question:** Given a house's size, rooms, age, location and distance from the city, what should its asking price be, and which factors move the price most?

- **Task:** regression. **Target variable:** `Price_Lakh` (asking price in lakh BDT).
- **Success metric:** **RMSE in lakh** ("typically off by ± X lakh"), with MAE and R².
- **Why RMSE:** it is in the same unit as the price, so agents understand it directly, and it punishes large mistakes more, which matter most in pricing. R² shows how much of the price differences the model explains.

## 2. Data dictionary
| Column | Meaning | Type | Unit | Notes |
|---|---|---|---|---|
| HouseID | Listing identifier | id | – | Not a feature |
| Location | City | category | Dhaka / Chattogram / Sylhet / Rajshahi | Text → one-hot |
| Area_sqft | Built-up floor area | number | sq ft | |
| Bedrooms | Bedrooms | count | rooms | |
| Bathrooms | Bathrooms | count | rooms | |
| HouseAge_years | Age of the house | number | years | |
| Garage | Garage | binary / count | 0/1 | |
| GardenArea_sqft | Garden size | number | sq ft | **30 missing**; 0 = no garden |
| DistanceToCity_km | Distance to city centre | number | km | **20 missing** |
| Price_Lakh | Asking price | number | lakh BDT | **Target** |

## 3. Data audit & cleaning

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip()
df["Location"] = df["Location"].astype(str).str.strip()
if not pd.api.types.is_numeric_dtype(df["Garage"]):
    df["Garage"] = df["Garage"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0, "y": 1, "n": 0, "1": 1, "0": 0})
print("Shape:", df.shape)
print("\nMissing:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum(), "| Duplicate HouseID:", df["HouseID"].duplicated().sum())
print("\nLocations:", df["Location"].value_counts().to_dict())
print("GardenArea = 0:", (df["GardenArea_sqft"] == 0).sum())
df.describe()

In [ ]:
for col in ["Area_sqft", "Price_Lakh", "GardenArea_sqft", "DistanceToCity_km", "HouseAge_years"]:
    q1, q3 = df[col].quantile([0.25, 0.75]); lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    print(f"{col:18s}: {((df[col] < lo) | (df[col] > hi)).sum()} outliers, range {df[col].min()}–{df[col].max()}")

**Decision log** (✍️ check the numbers against the outputs above)
| Issue | Decision | Reason |
|---|---|---|
| **GardenArea_sqft: 30 missing** | Fill with the **training-set median** + flag column `Garden_missing` | "No garden" is already recorded as **0**, so a blank means *unknown*, not "no garden" |
| **DistanceToCity_km: 20 missing** | Fill with the **median of the same Location** (training set) | Distance depends on the city, so a city median is more accurate than one overall median |
| Location is text (4 cities) | **One-hot encoding** | Mapping to 1–4 would invent an order and distances between cities |
| HouseID | Not a feature | An identifier carries no price information |
| Outliers | _✍️ kept / removed, and why_ | |
| Duplicates | _✍️ from the output_ | |

## 4. Exploratory data analysis

In [ ]:
sns.histplot(df["Price_Lakh"], kde=True); plt.title("Distribution of price (lakh)"); save_fig("01_price_distribution")

✍️ **Insight:** _write one or two sentences from the output above._

In [ ]:
sns.scatterplot(data=df, x="Area_sqft", y="Price_Lakh", hue="Location", alpha=0.7); plt.title("Price vs area"); save_fig("02_price_vs_area")
print("Correlation:", round(df["Area_sqft"].corr(df["Price_Lakh"]), 3))

✍️ **Insight:** _write one or two sentences from the output above._

In [ ]:
order = df.groupby("Location")["Price_Lakh"].median().sort_values().index
sns.boxplot(data=df, x="Location", y="Price_Lakh", order=order); plt.title("Price by location"); save_fig("03_price_by_location")
df.groupby("Location")["Price_Lakh"].agg(["count", "mean", "median"]).round(1)

✍️ **Insight:** _write one or two sentences from the output above._

In [ ]:
sns.scatterplot(data=df, x="DistanceToCity_km", y="Price_Lakh", alpha=0.6); plt.title("Price vs distance to city"); save_fig("04_price_vs_distance")

✍️ **Insight:** _write one or two sentences from the output above._

In [ ]:
sns.scatterplot(data=df, x="HouseAge_years", y="Price_Lakh", alpha=0.6); plt.title("Price vs house age"); save_fig("05_price_vs_age")

✍️ **Insight:** _write one or two sentences from the output above._

In [ ]:
sns.boxplot(data=df, x="Bedrooms", y="Price_Lakh"); plt.title("Price by number of bedrooms"); save_fig("06_price_by_bedrooms")

✍️ **Insight:** _write one or two sentences from the output above._

In [ ]:
num = df.drop(columns=["HouseID"]).select_dtypes("number")
plt.figure(figsize=(9, 7)); sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm"); plt.title("Correlation heatmap")
save_fig("07_correlation")

✍️ **Insight:** _write one or two sentences from the output above._ _Mention which features are correlated with each other (e.g. Area and Bedrooms)._

## 5. Feature preparation

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
garden_median = train_df["GardenArea_sqft"].median()
dist_by_loc, dist_all = train_df.groupby("Location")["DistanceToCity_km"].median(), train_df["DistanceToCity_km"].median()
locations = sorted(train_df["Location"].unique())

def prepare(data):
    X = data.drop(columns=["HouseID", "Price_Lakh"], errors="ignore").copy()
    X["Garden_missing"] = X["GardenArea_sqft"].isna().astype(int)
    X["GardenArea_sqft"] = X["GardenArea_sqft"].fillna(garden_median)
    X["DistanceToCity_km"] = X["DistanceToCity_km"].fillna(X["Location"].map(dist_by_loc)).fillna(dist_all)
    X["Location"] = pd.Categorical(X["Location"], categories=locations)
    return pd.get_dummies(X, columns=["Location"], drop_first=True, dtype=int)

X_train, y_train = prepare(train_df), train_df["Price_Lakh"]
X_test, y_test = prepare(test_df)[X_train.columns], test_df["Price_Lakh"]
print("Train:", X_train.shape, "Test:", X_test.shape, "| reference city:", locations[0], "| missing left:", int(X_train.isna().sum().sum()))

- **Encoding:** Location → one-hot (first city = reference). **Scaling:** not needed for linear regression predictions (coefficients stay in real units, which is easier to explain).
- **Engineered feature:** `Garden_missing` flag.
- **Leakage check:** the split happens **first**; fill values are learned from training data only. No input is computed from the price, and all inputs are known when a house is listed. HouseID is excluded.

## 6. Modelling

In [ ]:
def row(name, p): return {"Model": name, "R²": r2_score(y_test, p), "MAE (lakh)": mean_absolute_error(y_test, p), "RMSE (lakh)": np.sqrt(mean_squared_error(y_test, p))}
base = LinearRegression().fit(X_train[["Area_sqft"]], y_train)
pred_base = base.predict(X_test[["Area_sqft"]])
print(f"Baseline: Price = {base.intercept_:.2f} + {base.coef_[0]:.4f} × Area_sqft")
model = LinearRegression().fit(X_train, y_train); pred = model.predict(X_test)
tree = DecisionTreeRegressor(max_depth=5, random_state=42).fit(X_train, y_train)
results = pd.DataFrame([row("Baseline: area only", pred_base), row("Multiple regression", pred), row("Decision tree (stretch)", tree.predict(X_test))]).set_index("Model")

## 7. Evaluation

In [ ]:
results

In [ ]:
for m in results.index:
    print(f"{m}: explains {results.loc[m, 'R²']:.0%} of price differences; typically off by ±{results.loc[m, 'RMSE (lakh)']:.1f} lakh")

✍️ **Result:** _compare baseline vs multiple regression in business terms ("typically off by ± X lakh instead of ± Y")._

In [ ]:
resid = y_test - pred
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(pred, resid, alpha=0.6); ax[0].axhline(0, color="red", ls="--"); ax[0].set(xlabel="Predicted", ylabel="Residual", title="Residual plot")
ax[1].scatter(y_test, pred, alpha=0.6); l = [y_test.min(), y_test.max()]; ax[1].plot(l, l, "r--"); ax[1].set(xlabel="Actual", ylabel="Predicted", title="Actual vs predicted")
plt.tight_layout(); save_fig("08_residuals")

✍️ **Residual comment:** _random scatter around 0, or a funnel/curve? Which houses does the model struggle with?_

In [ ]:
worst = test_df.assign(Predicted=pred.round(1), Error=resid.round(1))
worst.reindex(resid.abs().sort_values(ascending=False).index).head(5)

✍️ **Error analysis:** _describe the 5 worst houses (city, size, age) and a likely reason._

In [ ]:
coef = pd.DataFrame({"Feature": X_train.columns, "Coefficient (lakh per unit)": model.coef_})
coef["Effect of +1 std (lakh)"] = coef.iloc[:, 1] * X_train.std().values
coef = coef.sort_values("Effect of +1 std (lakh)", key=abs, ascending=False); print("Intercept:", round(model.intercept_, 2))
c = dict(zip(X_train.columns, model.coef_))
print(f"• Every extra 100 sqft adds about {c['Area_sqft'] * 100:.2f} lakh")
print(f"• Each extra year of age changes price by {c['HouseAge_years']:.2f} lakh")
print(f"• Each extra km from the city changes price by {c['DistanceToCity_km']:.2f} lakh")
for k in [k for k in c if k.startswith("Location_")]: print(f"• {k[9:]} vs {locations[0]}: {c[k]:+.2f} lakh")
coef.round(3)

✍️ **Coefficients:** _interpret at least 3 in business language. Compare features with the standardised column; raw sizes depend on units._

In [ ]:
examples = pd.DataFrame([
    {"Location": locations[0], "Area_sqft": 1200, "Bedrooms": 3, "Bathrooms": 2, "HouseAge_years": 5, "Garage": 1, "GardenArea_sqft": 0, "DistanceToCity_km": 4},
    {"Location": locations[1], "Area_sqft": 1800, "Bedrooms": 4, "Bathrooms": 3, "HouseAge_years": 12, "Garage": 1, "GardenArea_sqft": 300, "DistanceToCity_km": 10},
    {"Location": locations[-1], "Area_sqft": 900, "Bedrooms": 2, "Bathrooms": 1, "HouseAge_years": 20, "Garage": 0, "GardenArea_sqft": 0, "DistanceToCity_km": 15}])
examples["Predicted_Price_Lakh"] = model.predict(prepare(examples)[X_train.columns]).round(1); examples

In [ ]:
#@title Price calculator (stretch) — change the values, then run
Location = "Dhaka" #@param ["Dhaka", "Chattogram", "Sylhet", "Rajshahi"] {allow-input: true}
Area_sqft = 1200 #@param {type:"number"}
Bedrooms = 3 #@param {type:"integer"}
Bathrooms = 2 #@param {type:"integer"}
HouseAge_years = 5 #@param {type:"number"}
Garage = 1 #@param {type:"integer"}
GardenArea_sqft = 0 #@param {type:"number"}
DistanceToCity_km = 5 #@param {type:"number"}
if Location not in locations: print("Use one of", locations)
else:
    h = pd.DataFrame([dict(Location=Location, Area_sqft=Area_sqft, Bedrooms=Bedrooms, Bathrooms=Bathrooms, HouseAge_years=HouseAge_years,
                           Garage=Garage, GardenArea_sqft=GardenArea_sqft, DistanceToCity_km=DistanceToCity_km)])
    print(f"Estimated price: {model.predict(prepare(h)[X_train.columns])[0]:.1f} lakh (typical error ±{results.loc['Multiple regression', 'RMSE (lakh)']:.1f})")

## 8. Conclusion & recommendation
✍️ _Fill in from the results:_
- **Answer:** the model estimates asking prices with R² = __ and a typical error of ± __ lakh (baseline: ± __). The biggest price drivers are __, __ and __.
- **Recommendation:** use the model as a **first estimate** that agents adjust, so two agents start from the same number.
- **Limitations:** 600 houses, course dataset (may be synthetic), no data on floor, condition, legal papers or road access; linear model assumes each factor adds a fixed amount.
- **Next steps:** add more listings and features, try tree-based models, retrain every quarter.

### Viva notes
1. **Location encoding?** One-hot; 1–4 would invent an order.
2. **R² meaning?** Share of price differences explained on unseen test houses.
3. **Largest coefficient = most important?** No: depends on units; compare standardised effects.
4. **Garden missing values?** 0 already means no garden, so blanks = unknown → median + flag.
5. **A house priced badly:** see the worst-5 table and explain why.